# Edinburgh Airbnb — Statistical Hypothesis Testing

This notebook tests the project's research hypotheses and reports confidence intervals and effect sizes alongside each statistical test.

A planned analysis of calendar-based pricing patterns was not possible because the Edinburgh snapshot's `calendar.csv` file contains no usable daily price data. The hypothesis was replaced with a neighbourhood-level price comparison based on validated listing price data.

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

sys.path.append(str(Path("..").resolve()))
from src.stats.effect_sizes import cohens_d, rank_biserial_r, epsilon_squared_kw, bootstrap_ci_mean

df = pd.read_csv("../data/processed/edinburgh/listings_master.csv", low_memory=False)
print(f"{len(df):,} listings loaded") 

4,925 listings loaded


## H1 — Entire home/apartment listings command higher prices than private rooms

**Null hypothesis (H0):**
There is no difference in the nightly price distributions of entire home/apartment listings and private room listings.

**Alternative hypothesis (H1):**
Entire home/apartment listings have higher nightly prices than private room listings.

In [4]:
h1_df = df[df["room_type"].isin(["Entire home/apt", "Private room"])].copy()
entire = h1_df.loc[h1_df["room_type"] == "Entire home/apt", "price_clean"].dropna()
private = h1_df.loc[h1_df["room_type"] == "Private room", "price_clean"].dropna()

print(f"Entire home/apt: n={len(entire)}, mean=£{entire.mean():.2f}, median=£{entire.median():.2f}, skew={entire.skew():.2f}")
print(f"Private room:    n={len(private)}, mean=£{private.mean():.2f}, median=£{private.median():.2f}, skew={private.skew():.2f}")

levene_stat, levene_p = stats.levene(entire, private)
print(f"\nLevene's test for equal variances: statistic={levene_stat:.2f}, p={levene_p:.4f}") 

Entire home/apt: n=3544, mean=£361.71, median=£186.00, skew=11.51
Private room:    n=1341, mean=£140.72, median=£87.00, skew=19.19

Levene's test for equal variances: statistic=14.86, p=0.0001


**Assumption check.** Both groups show substantial right skew, consistent with
the price distribution observed in EDA Part 1. Entire home/apartment listings
have a skewness of 11.51, while private rooms have an even stronger skew of
19.19, indicating that neither distribution is close to normal.

Levene's test was statistically significant (statistic = 14.86, p = 0.0001),
showing that the two groups also have unequal variances. Combined with the
strong non-normality, these results violate key assumptions of a standard
independent-samples t-test.

The Mann-Whitney U test was therefore used as the primary hypothesis test,
as it is more appropriate for heavily skewed data. Cohen's d is reported
alongside the test result as a secondary measure of effect size, but should
be interpreted cautiously because it assumes approximately normal
distributions.

In [5]:
u_stat, p_value = stats.mannwhitneyu(entire, private, alternative="greater")
effect_r = rank_biserial_r(u_stat, len(entire), len(private))
d = cohens_d(entire, private)

mean_e, lower_e, upper_e = bootstrap_ci_mean(entire.values)
mean_p, lower_p, upper_p = bootstrap_ci_mean(private.values)

print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_value:.2e}")
print(f"Rank-biserial r = {effect_r:.3f}  (primary effect size)")
print(f"Cohen's d        = {d:.3f}  (secondary - normality assumption not met, treat as approximate)")
print(f"\nEntire home/apt mean: £{mean_e:.2f}, 95% CI [£{lower_e:.2f}, £{upper_e:.2f}]")
print(f"Private room mean:    £{mean_p:.2f}, 95% CI [£{lower_p:.2f}, £{upper_p:.2f}]")

Mann-Whitney U: U=4086464, p=0.00e+00
Rank-biserial r = -0.720  (primary effect size)
Cohen's d        = 0.183  (secondary - normality assumption not met, treat as approximate)

Entire home/apt mean: £361.71, 95% CI [£317.51, £409.89]
Private room mean:    £140.72, 95% CI [£122.93, £165.19]


**H1 conclusion.** The Mann-Whitney U test found a statistically significant
difference between entire home/apartment listings and private room listings
(U = 4,086,464, p < 0.001). We therefore reject the null hypothesis and
conclude that entire home/apartment listings command higher prices than
private rooms in the Edinburgh Airbnb market.

The rank-biserial correlation magnitude was 0.720, indicating a large effect size, suggesting substantial separation between the two price distributions rather than a small difference that only appears significant because of the
large sample size. This is consistent with both the mean prices (£361.71 vs
£140.72) and median prices (£186 vs £87), which show that entire homes
occupy a distinctly higher pricing tier.

From a market perspective, room type is one of the strongest pricing signals
in the dataset and should be treated as a primary segmentation variable in
any pricing strategy. While there is still some overlap between the two
groups, the difference is large enough that comparing entire homes directly
with private rooms would obscure meaningful market structure.

## H2 — Superhost listings achieve higher review scores than non-superhosts

**Null hypothesis (H0):**
There is no difference in the review score distributions of superhost and non-superhost listings.

**Alternative hypothesis (H1):**
Superhost listings have higher review scores than non-superhost listings.

In [7]:
h2_df = df.dropna(subset=["review_scores_rating", "host_is_superhost"])
super_scores = h2_df.loc[h2_df["host_is_superhost"] == "t", "review_scores_rating"]
non_super_scores = h2_df.loc[h2_df["host_is_superhost"] == "f", "review_scores_rating"]

print(f"Superhost:     n={len(super_scores)}, mean={super_scores.mean():.3f}, skew={super_scores.skew():.2f}")
print(f"Non-superhost: n={len(non_super_scores)}, mean={non_super_scores.mean():.3f}, skew={non_super_scores.skew():.2f}")

levene_stat, levene_p = stats.levene(super_scores, non_super_scores)
print(f"\nLevene's test: statistic={levene_stat:.2f}, p={levene_p:.4f}") 

Superhost:     n=2166, mean=4.875, skew=-2.82
Non-superhost: n=2341, mean=4.690, skew=-3.86

Levene's test: statistic=465.36, p=0.0000


**Assumption check.** Review scores are bounded at 5 and heavily clustered near
the top of the scale, consistent with the ceiling effect observed in EDA Part 1.
Both groups also exhibit substantial negative skew (superhosts: -2.82,
non-superhosts: -3.86), indicating that the distributions are far from normal.

Levene's test was statistically significant (statistic = 465.36, p < 0.001),
showing that the variances of the two groups differ. Combined with the strong
departure from normality, this violates key assumptions of a standard
independent-samples t-test.

The Mann-Whitney U test was therefore used as the primary hypothesis test,
as it is more appropriate for comparing non-normal distributions. Any
parametric effect size measures should be interpreted cautiously given the
bounded and highly skewed nature of review score data.

In [8]:
u_stat, p_value = stats.mannwhitneyu(super_scores, non_super_scores, alternative="greater")
effect_r = rank_biserial_r(u_stat, len(super_scores), len(non_super_scores))
d = cohens_d(super_scores, non_super_scores)

mean_s, lower_s, upper_s = bootstrap_ci_mean(super_scores.values)
mean_ns, lower_ns, upper_ns = bootstrap_ci_mean(non_super_scores.values)

print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_value:.2e}")
print(f"Rank-biserial r = {effect_r:.3f}")
print(f"Cohen's d        = {d:.3f}  (secondary, normality not met)")
print(f"\nSuperhost mean:     {mean_s:.3f}, 95% CI [{lower_s:.3f}, {upper_s:.3f}]")
print(f"Non-superhost mean: {mean_ns:.3f}, 95% CI [{lower_ns:.3f}, {upper_ns:.3f}]")

Mann-Whitney U: U=3477576, p=4.35e-104
Rank-biserial r = -0.372
Cohen's d        = 0.687  (secondary, normality not met)

Superhost mean:     4.875, 95% CI [4.870, 4.880]
Non-superhost mean: 4.690, 95% CI [4.676, 4.704]


**H2 conclusion – statistical vs practical significance.** The Mann–Whitney U
test found a statistically significant difference in review scores between
superhost and non-superhost listings.
We therefore reject the null hypothesis and conclude that superhost listings
tend to receive higher review scores.

The effect size is moderate (r = 0.372), indicating that the difference is
not merely a consequence of the large sample size. Superhost listings have an
average review score of 4.875 compared with 4.690 for non-superhosts, a gap
of approximately 0.19 points. The 95% confidence intervals do not overlap
(superhosts: [4.870, 4.880]; non-superhosts: [4.676, 4.704]), providing
strong evidence that the difference is real.

However, the practical significance is more nuanced than the p-value alone
suggests. Both groups are rated highly on Airbnb's five-point scale, and a
difference of 0.19 points is unlikely to be the sole factor driving a guest's
booking decision. Even so, in a marketplace where review scores are heavily
clustered near the maximum value, a consistent advantage of nearly 0.2 points
represents a meaningful quality signal and supports the value of achieving
superhost status.

## H4 — Neighbourhood prices differ significantly 

**Null hypothesis (H0):**
The price distributions are the same across all neighbourhoods.

**Alternative hypothesis (H1):**
At least one neighbourhood has a different price distribution from the others.

In [9]:
neighbourhood_counts = df["neighbourhood_cleansed"].value_counts()
valid_neighbourhoods = neighbourhood_counts[neighbourhood_counts > 20].index
h4_df = df[df["neighbourhood_cleansed"].isin(valid_neighbourhoods)].dropna(subset=["price_clean"])

print(f"{len(valid_neighbourhoods)} neighbourhoods included (>20 listings each), {len(h4_df):,} listings total")
print(f"Excluded {df['neighbourhood_cleansed'].nunique() - len(valid_neighbourhoods)} neighbourhoods with 20 or fewer listings - too small for a stable group comparison")

groups = [g["price_clean"].values for _, g in h4_df.groupby("neighbourhood_cleansed")]

levene_stat, levene_p = stats.levene(*groups)
print(f"\nLevene's test across {len(groups)} groups: statistic={levene_stat:.2f}, p={levene_p:.4f}")

55 neighbourhoods included (>20 listings each), 4,283 listings total
Excluded 56 neighbourhoods with 20 or fewer listings - too small for a stable group comparison

Levene's test across 55 groups: statistic=1.23, p=0.1195


**Assumption check.** Fifty-five neighbourhoods were included in the analysis,
with neighbourhoods containing 20 or fewer listings excluded to avoid unstable
group estimates. Levene's test was not statistically significant
(statistic = 1.23, p = 0.1195), suggesting that differences in variance across
the neighbourhood groups are not severe enough to reject the equal-variance
assumption.

However, Airbnb prices are heavily right-skewed, with a small number of very
expensive listings creating a long upper tail. This departure from normality,
combined with the large number of groups being compared, makes a non-parametric
approach more appropriate. The Kruskal-Wallis test was therefore used as the
primary test of whether neighbourhood price distributions differ.

In [10]:
h_stat, p_value = stats.kruskal(*groups)
effect_epsilon = epsilon_squared_kw(h_stat, len(h4_df))

print(f"Kruskal-Wallis H: H={h_stat:.2f}, df={len(groups) - 1}, p={p_value:.2e}")
print(f"Epsilon-squared = {effect_epsilon:.3f}")

Kruskal-Wallis H: H=571.16, df=54, p=1.82e-87
Epsilon-squared = 0.133


**H4 conclusion.** The Kruskal-Wallis test found a statistically significant
difference in price distributions across Edinburgh neighbourhoods
(H = 571.16, df = 54, p = 1.82 × 10⁻⁸⁷). We therefore reject the null
hypothesis and conclude that neighbourhood location is associated with
meaningful differences in Airbnb listing prices.

The effect size was moderate (ε² = 0.133), indicating that neighbourhood
accounts for a meaningful proportion of the variation in listing prices,
although it is not the sole driver of price differences. As the
Kruskal-Wallis test only indicates that at least one neighbourhood differs
from the others, it does not identify the specific neighbourhood pairs
responsible for the result.

Rather than conducting a full set of pairwise post-hoc comparisons, the
neighbourhood rankings produced earlier in the star schema analysis and the
spatial patterns observed in the geographic EDA provide practical context for
interpreting these differences. Together, those results show that pricing is
not distributed evenly across the city and that neighbourhood should be
treated as an important factor in any pricing or market analysis. A formal
post-hoc procedure such as Dunn's test would be a reasonable extension for
future work if identifying the exact neighbourhood pairs of interest were
required.

## Summary

| Hypothesis                                 | Test           | Statistic     | p-value       | Effect size | Conclusion                                                                                        |
| ------------------------------------------ | -------------- | ------------- | ------------- | ----------- | ------------------------------------------------------------------------------------------------- |
| H1: Entire home > private room price       | Mann-Whitney U | U = 4,086,464 | < 0.001       | r = 0.720  | Reject H0. Entire home/apartment listings command significantly higher prices than private rooms. |
| H2: Superhost > non-superhost review score | Mann-Whitney U | U = 3,477,576 | 4.35 × 10⁻¹⁰⁴ | r = 0.372  | Reject H0. Superhost listings achieve significantly higher review scores than non-superhosts.     |
| H4: Price differs by neighbourhood         | Kruskal-Wallis | H = 571.16    | 1.82 × 10⁻⁸⁷  | ε² = 0.133  | Reject H0. Airbnb price distributions differ significantly across neighbourhoods.                 |
